# Bölüm 5c — BrickEconomy Öncelikli Örneklem Seçimi

**Amaç:** BrickEconomy günde 100 / dakikada 4 istekle sınırlı olduğu için tüm
setleri çekemiyoruz (bkz. [DATA_SOURCES.md](../DATA_SOURCES.md)). Bu not
defteri, ~2.200 setlik çekim kotasının **hangi setlere** ayrılacağını
belirliyor — henüz hiçbir BrickEconomy API çağrısı yapılmıyor, sadece
önceliklendirme mantığı ve seçilen örneklemin profili hazırlanıyor.

**Havuz:** Sadece **Brickset ile zaten eşleşmiş 6.614 set** arasından
seçim yapılıyor. Gerekçe: bu setler için retail price zaten var; BrickEconomy'den
`current_value` + `growth` + `forecast` eklemek onları hem Bölüm 2 (fiyat
tahmini) hem Faz 3 (retired-sonrası değer artışı) için **tam veri** haline
getiriyor — iki analiz aynı set üzerinde birleşiyor.

In [1]:
import pandas as pd
import numpy as np

sets_clean = pd.read_csv("../data/processed/sets_clean.csv")
brickset_prices = pd.read_csv("../data/processed/brickset_prices.csv")

merged = sets_clean.merge(brickset_prices, left_on="set_num", right_on="brickset_set_num", how="left")
ready = merged[merged["is_analysis_ready"] == True].copy()
pool = ready[ready["retail_price_us"].notna()].copy()
print(f"Havuz (Brickset ile eşleşen, is_analysis_ready): {len(pool)}")

Havuz (Brickset ile eşleşen, is_analysis_ready): 6614


## 1) "Üretim süresi" proxy metriği

**Varsayım:** LEGO, kıtlık yaratan bir lüks marka gibi davranmıyor — tam
tersine, **popüler/tercih edilen bir seti üretimde daha uzun süre tutuyor**,
az satan bir seti ise daha erken raftan kaldırıyor. Bu nedenle bir setin
`date_first_available`'dan `date_last_available`'a kadar geçen süresi
("üretim süresi"), o setin ne kadar tercih edildiğine dair makul bir proxy
olarak kullanılabilir.

Bu, Faz 4'te tanımlanan "üretim süresi" proxy metriğiyle **kavramsal olarak
aynı** — burada BrickEconomy önceliklendirmesi için, orada tema
ömrü/başarı analizinde kullanılabilir; ikisi çapraz besleniyor.

In [2]:
pool["date_first_available"] = pd.to_datetime(pool["date_first_available"], errors="coerce", utc=True)
pool["date_last_available"] = pd.to_datetime(pool["date_last_available"], errors="coerce", utc=True)
pool["production_days"] = (pool["date_last_available"] - pool["date_first_available"]).dt.days

has_days = pool["production_days"].notna()
print(f"Üretim süresi hesaplanabilen: {has_days.sum()} / {len(pool)}")
print(f"Eksik (en sona atılacak): {(~has_days).sum()} / {len(pool)}")

neg = pool[pool["production_days"] < 0]
print(f"Negatif üretim süresi (veri hatası şüphesi): {len(neg)}")

Üretim süresi hesaplanabilen: 6149 / 6614
Eksik (en sona atılacak): 465 / 6614
Negatif üretim süresi (veri hatası şüphesi): 0


## 2) Bileşik öncelik skoru

`combined_score = z(production_days) + z(num_parts)` — ikisi de standardize edilip toplanıyor (basit, yorumlanabilir). Üretim süresi eksik olan setler skordan bağımsız olarak listenin en sonuna atılıyor.

In [3]:
z_days = (pool.loc[has_days, "production_days"] - pool.loc[has_days, "production_days"].mean()) / pool.loc[has_days, "production_days"].std()
pool["z_production_days"] = np.nan
pool.loc[has_days, "z_production_days"] = z_days
pool["z_num_parts"] = (pool["num_parts"] - pool["num_parts"].mean()) / pool["num_parts"].std()
pool["combined_score"] = pool["z_production_days"] + pool["z_num_parts"]
pool["has_days"] = has_days

# has_days=True önce (True > False sıralamada), sonra combined_score azalan,
# eksik grup içinde tiebreak olarak num_parts azalan
priority_order = pool.sort_values(by=["has_days", "combined_score", "num_parts"], ascending=[False, False, False])
top2200 = priority_order.head(2200)

print(f"Seçilen 2.200 içinde üretim-süresi-dolu olan: {top2200['has_days'].sum()}")
print(f"Seçilen 2.200 içinde eksik (sona atılmış ama yine de kotaya giren): {(~top2200['has_days']).sum()}")
print()
print(f"Medyan num_parts: {top2200['num_parts'].median():.0f}")
print(f"Medyan üretim süresi: {top2200.loc[top2200['has_days'], 'production_days'].median():.0f} gün (~{top2200.loc[top2200['has_days'], 'production_days'].median()/30:.1f} ay)")
print(f"Medyan yıl: {top2200['year'].median():.0f}")

Seçilen 2.200 içinde üretim-süresi-dolu olan: 2200
Seçilen 2.200 içinde eksik (sona atılmış ama yine de kotaya giren): 0

Medyan num_parts: 597
Medyan üretim süresi: 742 gün (~24.7 ay)
Medyan yıl: 2019


## 3) Seçilen 2.200 setin tema dağılımı

In [4]:
top_themes = top2200["theme_name"].value_counts()
print(f"Seçilen 2.200 set, {top2200['theme_name'].nunique()} farklı temaya yayılıyor.")
print()
print("En sık 15 tema:")
print(top_themes.head(15))

Seçilen 2.200 set, 165 farklı temaya yayılıyor.

En sık 15 tema:
theme_name
Star Wars                195
Technic                  148
Friends                  122
Creator 3-in-1           114
Ninjago                   91
Minecraft                 69
Harry Potter              58
Police                    51
Classic                   43
Town                      42
LEGO Ideas and CUUSOO     41
Architecture              38
Monkie Kid                38
City                      37
Speed Champions           37
Name: count, dtype: int64


**Gözlem:** Bu bileşik önceliklendirme (üretim süresi + parça sayısı) ile en
çok öne çıkan temalar **Star Wars (195), Technic (148), Friends (122),
Creator 3-in-1 (114) ve Ninjago (91)** oluyor — bunları Minecraft, Harry
Potter, Police, Classic, Town gibi temalar takip ediyor. Toplamda 165 farklı
temaya yayılmış durumda (tek birkaç temaya aşırı yığılma yok). Bu, "en
revaçta temalar hangileri" sorusuna erken/kaba bir cevap veriyor — hem uzun
üretimde kalmış hem büyük setler üreten temalar bunlar. Bölüm 1/4'teki tema
başarı analizinde bu sıralamanın tutarlı çıkıp çıkmadığı ayrıca
doğrulanabilir.

## 4) Minifigür kotası (~800) — BrickEconomy endpoint durumu

**BrickEconomy'nin ayrı bir minifigür endpoint'i VAR:** `GET
/minifig/{minifigNumber}` (ör. `"sw0509"`), `current_value_new` ve
`price_events_new` döndürüyor (OpenAPI şemasından doğrulandı).

**Ancak format uyuşmazlığı var:** BrickEconomy'nin minifig kodları
(tema-önekli, `"twn402"`, `"sw0509"` gibi) Rebrickable'ın `fig_num`
formatıyla (`"fig-000001"`) uyumlu değil, doğrudan bir dönüşüm yok.

**Önceliklendirme için — henüz karar verilmedi, seçenekler:**

1. **Set-yanıtlarından türetme:** Yukarıdaki 2.200 setin BrickEconomy
   yanıtlarındaki `minifigs: [...]` alanlarından toplanan benzersiz kodlar
   (doğru format garanti, zaten çekilen setlerle ilişkili minifigürler
   önceliklenir).
2. **Rastgele örneklem:** rpmoo ve Brickset'te minifig-bazlı fiyat/nadirlik
   verisi olmadığı doğrulandı (rpmoo sadece set fiyatı çekmiş; Brickset'in
   `getSets` çıktısı sadece minifig *kodlarını* veriyor, fiyat değil) —
   yani "en pahalı/nadir" diye önceliklendirecek bağımsız bir sinyal şu an
   elimizde yok. Bu durumda Rebrickable'ın `minifigs.csv`'sindeki mevcut
   minifigürlerden rastgele bir örneklemle başlamak tek pratik alternatif.

Bu iki seçenek arasında karar **kullanıcıya bırakıldı** — henüz uygulanmadı.

## 5) Genişletilmiş öncelik sırası — havuzun tamamı (6.614 set)

**Karar (2026-09-13):** BrickEconomy üyeliği ~13 Kasım 2026'ya uzatıldı ve günlük kota 100'e
çıkarıldı. Bu, çekimin başladığı günden itibaren ~6.000 isteklik bir bütçe demek — ilk plan
(2.200 set + ~800 minifig ≈ 3.000 istek) bunun yarısını kullanıyordu.

Sabit bir yeni hedef sayı seçmek yerine **havuzun tamamı aynı bileşik skorla sıralanıyor** ve
sırayla çekiliyor. Üyelik bittiğinde nerede kalınırsa kalınsın, elde her zaman *en öncelikli N set*
olur. Çekim sırası (`scripts/fetch_brickeconomy_daily.py`):

1. Setler, sıra 1–2.200 (ilk plan — değişmedi, önbellekteki setler aynen geçerli)
2. ~800 minifig (1. adımdaki setlerin `minifigs` alanlarından)
3. Setler, sıra 2.201–6.614 (genişleme — süre biterse buradan kırpılır)

İlk 2.200 satır, mevcut `brickeconomy_priority_sets.csv` ile birebir aynı olmak zorunda; aşağıdaki
hücre bunu doğrulamadan dosyayı yazmıyor.


In [5]:
from pathlib import Path

PRIORITY_PATH = Path("../data/processed/brickeconomy_priority_sets.csv")
EXPORT_COLS = ["priority_rank", "set_num", "name", "theme_name", "year", "num_parts", "production_days", "combined_score"]

ranked = priority_order.reset_index(drop=True).copy()
ranked["priority_rank"] = ranked.index + 1
ranked = ranked[EXPORT_COLS]

existing = pd.read_csv(PRIORITY_PATH)
if len(existing) > 2200:
    existing = existing.head(2200)
head_matches = (ranked["set_num"].head(2200).values == existing["set_num"].values).all()
print(f"İlk 2.200 set, dondurulmuş listeyle aynı sırada: {head_matches}")
assert head_matches, "Sıralama değişmiş — önbellekteki çekimle uyumsuz olur, dosya yazılmadı."

ranked.to_csv(PRIORITY_PATH, index=False)
print(f"Yazıldı: {PRIORITY_PATH.name} — {len(ranked)} set")


İlk 2.200 set, dondurulmuş listeyle aynı sırada: True
Yazıldı: brickeconomy_priority_sets.csv — 6614 set


In [6]:
extension = priority_order.iloc[2200:]
first = priority_order.head(2200)

profile = pd.DataFrame({
    "sıra 1–2.200": [len(first), first["num_parts"].median(), first["year"].median(), (~first["has_days"]).sum(), first["theme_name"].nunique()],
    "sıra 2.201–6.614": [len(extension), extension["num_parts"].median(), extension["year"].median(), (~extension["has_days"]).sum(), extension["theme_name"].nunique()],
}, index=["set sayısı", "medyan num_parts", "medyan yıl", "üretim süresi eksik", "tema sayısı"])
profile


,sıra 1–2.200,sıra 2.201–6.614
set sayısı,2200.0,4414.0
medyan num_parts,597.0,143.0
medyan yıl,2019.0,2016.0
üretim süresi eksik,0.0,465.0
tema sayısı,165.0,222.0


**Gözlem:** Genişleme dilimi belirgin biçimde farklı bir profil taşıyor — daha küçük (medyan parça
sayısı ~600 yerine ~140) ve biraz daha eski (medyan yıl 2019 yerine 2016) setler; üretim süresi
eksik olan 465 setin tamamı da bu dilimin en sonunda. Bu, örneklemi büyük/flagship setlerin ötesine
genişletiyor (Brickset eşleşmesindeki yanlılığı kısmen dengeliyor), ama analizlerde **sıra dilimi
bir değişken olarak tutulmalı**: 2.200 sonrası setler seçim mantığı gereği sistematik olarak farklı.


## 6) Brickset fiyatı olmayan setler — kalan kotanın kullanımı

**Karar (2026-11-10):** Üyelik 8 Aralık 2026'ya kadar uzadı. 6.614 setlik öncelik listesi ~26 Kasım'da bitiyor;
7 Aralık'a kadar kalan ~1.100 istek, **Brickset'te liste fiyatı hiç olmayan** analiz-hazır setlere harcanıyor.

**Gerekçe:** Bölüm 2'nin fiyat verisi sadece Brickset eşleşmesinden geliyor ve bu eşleşme 2015 sonrasına ve büyük setlere
yığılıyor (notebook 05, 08). BrickEconomy de `retail_price_us` veriyor; çekilen 5.073 sette Brickset fiyatıyla **%97,2
birebir aynı**. Yani Brickset'in kapsamadığı setler için güvenilir bir ikinci fiyat kaynağı olabilir.

**Seçim (fiyata bakılmadan, önceden belirlenen ölçütle):**
1. 2005–2014, ≥50 parça (723 set — tamamı sığıyor)
2. 1990–2004, ≥50 parça (kalan kota kadar)

Her dilim içinde parça sayısına göre büyükten küçüğe. ≥50 parça şartı, tek parça/aksesuar paketlerini dışarıda tutuyor.

**Belirsizlik:** BrickEconomy'nin bu setler için fiyat verip vermediği bilinmiyor (çekilen 2005–2009 setlerinde %100, ama o
setler Brickset'te de fiyatlıydı). Çekim script'i her çalıştırmada bu aşamanın fiyat doluluk oranını loga yazıyor.


In [7]:
UNPRICED_PATH = Path("../data/processed/brickeconomy_unpriced_sets.csv")

priced_set_nums = set(brickset_prices.loc[brickset_prices["retail_price_us"].notna(), "brickset_set_num"])
unpriced = ready[~ready["set_num"].isin(priced_set_nums) & (ready["num_parts"] >= 50)].copy()
unpriced["unpriced_tier"] = np.select(
    [unpriced["year"].between(2005, 2014), unpriced["year"].between(1990, 2004)], [1, 2], 0
)
unpriced = unpriced[unpriced["unpriced_tier"] > 0].sort_values(["unpriced_tier", "num_parts"], ascending=[True, False])
unpriced["unpriced_rank"] = range(1, len(unpriced) + 1)

assert not set(unpriced["set_num"]) & set(ranked["set_num"]), "Öncelik listesiyle çakışma var"
unpriced[["unpriced_rank", "unpriced_tier", "set_num", "name", "theme_name", "year", "num_parts"]].to_csv(UNPRICED_PATH, index=False)
print(f"Yazıldı: {UNPRICED_PATH.name} — {len(unpriced)} set")
unpriced.groupby("unpriced_tier").agg(set_sayisi=("set_num", "size"), medyan_parca=("num_parts", "median"),
                                      medyan_yil=("year", "median"), tema_sayisi=("theme_name", "nunique"))


Yazıldı: brickeconomy_unpriced_sets.csv — 2581 set


,set_sayisi,medyan_parca,medyan_yil,tema_sayisi
unpriced_tier,,,,
1,723,129.0,2009.0,89
2,1858,148.0,1999.0,139
